# Customer Satisfaction
**Company:** Atlassian (GothamLoop question bank) · **Category:** Coding · **Tags:** Onsite Loop, Hash Tables, Sorting · **Difficulty/Frequency:** Rare (2/10)


## Concepts

**What this problem is really testing:**
- Incremental aggregation — keep a running sum + count, not the full raw history
- Nested hash maps, to add a second grouping level (month)
- Deterministic multi-key sorting

**Why each one shows up here:**
- "Average rating per agent" only ever needs a running total and a count. You *could* store every individual rating, but that wastes memory and forces a full recompute (O(n)) on every query.
- Adding "per month" is just a second key layered in front of the same aggregate — nothing fundamentally new.
- The tie-breaking and export requirements are just about making the *output* of that aggregate predictable and self-explanatory.

**The one idea to hold onto:** never store more than the aggregate the query actually needs. Since `average = total / count`, a `[total, count]` pair is a complete summary — you can always recover the average from it, and you can always combine two of these pairs just by adding the matching numbers together.

---

### Quick primers — the building blocks used below

**What is a Hash Map?**
- A hash map (Python `dict`) stores key → value pairs by hashing the key to a slot, giving **O(1) average** insert/lookup/delete.
- **In Python:** `collections.defaultdict` automatically creates a default value (here, `[0, 0]`) the first time a missing key is accessed — no need to manually check `if key not in d: d[key] = ...` before every insert.

**Nested hash maps, for a second grouping level.**
- `{month: {agent_id: [total, count]}}` just means "group by month, then group by agent inside each month" — the outer dict's values are themselves dicts.
- This generalizes to as many levels as you need — each extra level just costs one more dict lookup, still O(1) per level.

**Storing aggregates instead of raw data.**
- Keeping `[sum, count]` per key means:
  - Inserting a new rating is O(1) — just update two numbers.
  - The average is available on demand in O(1).
- Compare that to storing a growing list of raw ratings and re-summing the whole list every time an average is needed.


## Problem Statement

- `accept_rating(agent_id, rating, month)` -- record a 1-5 rating for an agent in a given month.
- `get_all_agents_sorted(month=None)` -- agents and their average rating, highest to lowest (all-time if `month` is omitted).
- **Tie handling:** decide and implement a deterministic tiebreak.
- **Monthly best agents:** retrieve the ranking for any given month.
- **Export:** dump every agent's per-month average AND total rating **count** (not just the total sum) in CSV or JSON.


### Approach 1 -- Naive (store raw ratings, aggregate on every query)

**Idea:** append every `(agent_id, rating, month)` triple to a flat list. `accept_rating` is O(1) (a list append). Every query re-scans the entire history to build the aggregates before it can answer.

**Time complexity:** O(1) for `accept_rating`; **O(r)** for every query, where r is the total number of ratings ever recorded -- re-derives the same aggregates from scratch every single call.

**Space complexity:** O(r) -- one entry per rating, forever.


In [ ]:
from collections import defaultdict
from typing import Dict, List, Tuple, Optional


class RatingSystemNaive:
    def __init__(self) -> None:
        self._raw: List[Tuple[str, int, str]] = []   # (agent_id, rating, month) -- unaggregated

    def accept_rating(self, agent_id: str, rating: int, month: str) -> None:
        if not (1 <= rating <= 5):
            raise ValueError("Rating must be between 1 and 5")
        self._raw.append((agent_id, rating, month))

    def get_all_agents_sorted(self, month: Optional[str] = None) -> List[Tuple[str, float]]:
        totals: Dict[str, List[int]] = defaultdict(lambda: [0, 0])
        for agent_id, rating, m in self._raw:            # re-scans EVERY rating, every call
            if month is not None and m != month:
                continue
            totals[agent_id][0] += rating
            totals[agent_id][1] += 1
        results = [(a, t / c) for a, (t, c) in totals.items()]
        results.sort(key=lambda x: (-x[1], x[0]))
        return results


### Approach 2 -- Optimal (nested running aggregates)

**Idea:** `{month: {agent_id: [total, count]}}`. `accept_rating` is O(1): look up (or auto-create via `defaultdict`) the `[total, count]` pair and update both numbers in place -- no rescanning. Queries combine only the relevant months' aggregates (one month, or all of them) and sort once.

**Time complexity:** O(1) for `accept_rating`. `get_all_agents_sorted` is O(months_considered * agents_per_month + m log m) for m distinct agents in scope -- no dependency on the *total number of ratings ever recorded*, only on how many distinct agent-month buckets exist.

**Space complexity:** O(months * agents) -- one `[total, count]` pair per agent-month combination that actually received a rating.


In [ ]:
class CustomerSupportRatingSystem:
    def __init__(self) -> None:
        # {month: {agent_id: [total_rating, count]}}
        self._ratings: Dict[str, Dict[str, List[int]]] = defaultdict(lambda: defaultdict(lambda: [0, 0]))

    def accept_rating(self, agent_id: str, rating: int, month: str) -> None:
        if not (1 <= rating <= 5):
            raise ValueError("Rating must be between 1 and 5")
        stats = self._ratings[month][agent_id]
        stats[0] += rating          # running total -- O(1), no rescanning
        stats[1] += 1                # running count

    def get_all_agents_sorted(self, month: Optional[str] = None) -> List[Tuple[str, float]]:
        months = [month] if month is not None else list(self._ratings.keys())
        totals: Dict[str, List[int]] = defaultdict(lambda: [0, 0])
        for m in months:
            for agent_id, (total, count) in self._ratings.get(m, {}).items():
                totals[agent_id][0] += total
                totals[agent_id][1] += count

        results = [(a, t / c) for a, (t, c) in totals.items()]
        results.sort(key=lambda x: (-x[1], x[0]))       # descending average, then agent_id ascending
        return results

    def get_best_agents_by_month(self) -> Dict[str, List[Tuple[str, float]]]:
        return {m: self.get_all_agents_sorted(month=m) for m in self._ratings}

    def export_ratings(self, fmt: str = "csv") -> str:
        """Export every (month, agent) average AND count, unsorted."""
        rows = [
            (month, agent_id, total / count, count)
            for month, agents in self._ratings.items()
            for agent_id, (total, count) in agents.items()
        ]
        if fmt.lower() == "json":
            import json
            return json.dumps(
                [{"month": m, "agent_id": a, "average_rating": round(avg, 2), "total_rating_count": c}
                 for m, a, avg, c in rows],
                indent=2,
            )
        lines = ["month,agent_id,average_rating,total_rating_count"]
        lines += [f"{m},{a},{avg:.2f},{c}" for m, a, avg, c in rows]
        return "\n".join(lines)


## Verification

Check both implementations agree, the tie-break is deterministic, and the edge cases the Talking Points and Follow-ups call out.

In [ ]:
def load(system):
    system.accept_rating("agent_a", 5, "2024-01")
    system.accept_rating("agent_a", 4, "2024-01")     # agent_a: 9/2 = 4.5 in Jan
    system.accept_rating("agent_b", 4, "2024-01")     # agent_b: 4/1 = 4.0 in Jan
    system.accept_rating("agent_b", 4, "2024-02")     # agent_b: 4/1 = 4.0 in Feb
    system.accept_rating("agent_c", 3, "2024-02")     # agent_c: 3/1 = 3.0 in Feb
    return system

naive = load(RatingSystemNaive())
optimal = load(CustomerSupportRatingSystem())

assert naive.get_all_agents_sorted() == optimal.get_all_agents_sorted()
assert optimal.get_all_agents_sorted() == [("agent_a", 4.5), ("agent_b", 4.0), ("agent_c", 3.0)]
assert optimal.get_all_agents_sorted(month="2024-01") == [("agent_a", 4.5), ("agent_b", 4.0)]
assert optimal.get_all_agents_sorted(month="2024-02") == [("agent_b", 4.0), ("agent_c", 3.0)]

# Deterministic tie-break: two agents with the SAME average must sort by agent_id ascending
tied = CustomerSupportRatingSystem()
tied.accept_rating("zeta", 5, "2024-01")
tied.accept_rating("alpha", 5, "2024-01")
assert tied.get_all_agents_sorted() == [("alpha", 5.0), ("zeta", 5.0)]   # alphabetical, not insertion order

# Monthly best-agents view
best_by_month = optimal.get_best_agents_by_month()
assert best_by_month["2024-01"] == [("agent_a", 4.5), ("agent_b", 4.0)]
assert best_by_month["2024-02"] == [("agent_b", 4.0), ("agent_c", 3.0)]

# Export: both formats include the COUNT, not just the average (a Talking Point requirement)
csv_out = optimal.export_ratings("csv")
assert "agent_a,4.50,2" in csv_out                # 2 ratings averaging to 4.5
json_out = optimal.export_ratings("json")
import json as _json
parsed = _json.loads(json_out)
row = next(r for r in parsed if r["agent_id"] == "agent_a" and r["month"] == "2024-01")
assert row["average_rating"] == 4.5 and row["total_rating_count"] == 2

# Input validation
try:
    optimal.accept_rating("agent_x", 6, "2024-01")
    assert False, "should have raised"
except ValueError:
    pass

# Empty system
empty = CustomerSupportRatingSystem()
assert empty.get_all_agents_sorted() == []
assert empty.export_ratings() == "month,agent_id,average_rating,total_rating_count"

print("All checks passed.")


## Discussion -- remaining follow-up directions

- **Millions of ratings/day, hourly export.** The in-memory nested dict still works for the *aggregates* (they never grow with ratings volume, only with agent count x month count) -- the real bottleneck would be durability and multi-process access, which points at a real database (or a periodically-flushed in-memory layer) rather than an algorithmic change.
- **Weighted/decaying ratings (recent ratings count more).** Storing `[total, count]` alone can't support this -- you'd need to store enough to recompute a decayed average, e.g. a running `(weighted_sum, weight_sum)` pair where weight is a function of a stored rating timestamp, recomputed relative to "now" at query time, or a sliding window that only sums ratings within the last N days.
- **Deactivated agents excluded from rankings.** Add an `active: Dict[str, bool]` set/map and filter in `get_all_agents_sorted` -- a pure filtering change, no aggregate-structure change needed.
- **An agent with zero ratings in a queried month.** Decide explicitly: omit them (current behavior -- they simply have no entry in that month's dict), show `0.0`, or show `"N/A"` -- and apply that choice consistently across `get_all_agents_sorted`, `get_best_agents_by_month`, and `export_ratings`.
- **Adding new export formats easily.** A `format -> formatter_function` dispatch dict (or a small strategy-pattern class per format) keeps `export_ratings` from growing an ever-longer `if/elif` chain as XML, Parquet, etc. get added -- shown below.


In [ ]:
def _to_csv(rows):
    lines = ["month,agent_id,average_rating,total_rating_count"]
    lines += [f"{m},{a},{avg:.2f},{c}" for m, a, avg, c in rows]
    return "\n".join(lines)


def _to_json(rows):
    import json
    return json.dumps(
        [{"month": m, "agent_id": a, "average_rating": round(avg, 2), "total_rating_count": c}
         for m, a, avg, c in rows],
        indent=2,
    )


EXPORT_FORMATTERS = {"csv": _to_csv, "json": _to_json}   # add "xml": _to_xml, etc. with no other changes


def export_ratings_extensible(system: CustomerSupportRatingSystem, fmt: str = "csv") -> str:
    rows = [
        (month, agent_id, total / count, count)
        for month, agents in system._ratings.items()
        for agent_id, (total, count) in agents.items()
    ]
    formatter = EXPORT_FORMATTERS.get(fmt.lower())
    if formatter is None:
        raise ValueError(f"Unsupported export format: {fmt}")
    return formatter(rows)


assert export_ratings_extensible(optimal, "csv") == optimal.export_ratings("csv")
print("Extensible export dispatch works as expected.")


## Empirical complexity check

The optimal system's `accept_rating` is O(1) regardless of history size; the naive version's `get_all_agents_sorted` is O(r) in the total ratings recorded. We compare: **N calls to `accept_rating` followed by 1 query**, growing N -- optimal total work should stay close to linear in N (dominated by the N inserts), while naive's single query cost alone already scales with N, on top of the same N inserts.

| Growth when n doubles | Implies |
|---|---|
| ~2x | linear |


In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

AGENTS = [f"agent{i}" for i in range(20)]


def run_naive(n):
    s = RatingSystemNaive()
    for i in range(n):
        s.accept_rating(AGENTS[i % 20], (i % 5) + 1, "2024-01")
    s.get_all_agents_sorted()          # one query AFTER n inserts -- O(n) on top of the n inserts


def run_optimal(n):
    s = CustomerSupportRatingSystem()
    for i in range(n):
        s.accept_rating(AGENTS[i % 20], (i % 5) + 1, "2024-01")
    s.get_all_agents_sorted()          # O(agents) -- independent of n


def make_worst_case(n):
    return (n,)


solutions = {"naive (rescan on query)": run_naive, "optimal (running aggregates)": run_optimal}
sizes = [4000, 8000, 16000, 32000]
benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Store the aggregate, not the history.** `[sum, count]` is a complete, O(1)-updatable, O(1)-to-average summary -- there's rarely a reason to keep every raw data point around just to compute a mean.
- **Nested dicts are how you add a grouping dimension without redesigning.** `{agent: stats}` -> `{month: {agent: stats}}` is the same pattern one level deeper; the query layer (sum across the relevant months) is the only new code.
- **Deterministic sort keys are a correctness requirement, not polish.** `(-average, agent_id)` isn't just "nicer output" -- without a tiebreak, two runs of the same program can legally produce different orders for tied agents, which breaks tests and confuses users.
- **Exports should be self-describing.** A total alone (45) is ambiguous; pairing it with a count (45 from 9 ratings vs. 45 from 45 ratings) lets any downstream consumer recover the full picture, including recomputing the average themselves.
- **Related problems:** any leaderboard/dashboard aggregation (top sellers by region-and-month, click-through rate by campaign-and-day), streaming mean/variance (Welford's algorithm generalizes "running aggregate" to more statistics).
- **Common pitfalls:** recomputing aggregates from raw history on every read instead of maintaining them incrementally; sorting only by the computed value and letting ties fall to unstable/insertion order; exporting an average without the sample size backing it.
